# Python Regex Exercises: 30 Coding Problems with Solutions

A practice notebook on the `re` module — quantifiers, character classes, anchors, groups, extraction, and substitution, with real-world tasks like email extraction, date reformatting, and IP cleaning — each with a concept note, a hint, a solution, and an explanation.

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/python-regex-exercises/).*

---

## Concepts you'll need

This set covers Python's `re` module — pattern matching, extraction, and substitution.

- **Quantifiers** — `*` (zero or more), `+` (one or more), `?` (zero or one), `{n}` (exactly n), `{m,n}` (between m and n, inclusive on both ends). All apply to whatever comes immediately before them.
- **Character classes** — `[a-z]`, `[A-Z0-9]`, etc. define a set of acceptable characters for one position. Shorthands: `\d` (digit, `[0-9]`), `\w` (word character, `[a-zA-Z0-9_]`), `\s` (whitespace). Inside `[...]`, a dot loses its special meaning and becomes literal.
- **The dot `.`** — outside a character class, matches any single character except a newline. `.* ` is the classic "anything in between" pattern.
- **Anchors** — `^` (start of string), `$` (end of string), `\b` (word boundary — the position between a word character and a non-word character, not a character itself).
- **Matching functions** — `re.fullmatch()` (the *entire* string must match — strictest), `re.match()` (must match starting at position 0, but can leave trailing content), `re.search()` (finds a match anywhere in the string, returns the first one).
- **Extraction functions** — `re.findall()` (returns a list of matched strings, or tuples if the pattern has groups), `re.finditer()` (returns a lazy iterator of full match objects, giving access to `.start()`, `.end()`, `.span()`, `.group()` — more memory-efficient and more informative than `findall()`).
- **Groups** — `(...)` captures part of a match for later reference; `(?:...)` groups without capturing (doesn't affect `findall()`'s output). Backreferences in a `re.sub()` replacement string (`\1`, `\2`, ...) refer to captured groups, letting you reorder or reuse matched text.
- **Substitution** — `re.sub(pattern, replacement, string)` replaces every match. The replacement can be a plain string (with `\1`-style backreferences) or a *callable* that receives the match object and returns the replacement text — essential when the replacement needs computation, like `lambda m: str(int(m.group()))`.
- **Raw strings** — regex patterns are almost always written as `r"..."` so that backslashes (needed for `\d`, `\b`, `\w`, etc.) aren't first interpreted as Python string escapes.
- **`re.escape()`** — escapes any regex-special characters in a plain string, essential whenever a search term comes from user input rather than being a hardcoded literal.

Each exercise below gives a problem, a hint, a solution, and an explanation.

## Exercise 1. Check Allowed Characters

**Concept:** re.fullmatch() for strict whole-string validation

**Problem:** Verify a string contains only letters and digits.

**Given:**
```
text = "Hello123"
```

**Expected Output:**
```
Valid: contains only alphanumeric characters
```

**Hint:** fullmatch() requires the ENTIRE string to match, unlike search() which accepts a partial match anywhere.

In [ ]:
import re

text = "Hello123"

if re.fullmatch(r"[a-zA-Z0-9]+", text):
    print("Valid: contains only alphanumeric characters")
else:
    print("Invalid: contains non-alphanumeric characters")

**Explanation:** [a-zA-Z0-9] is a character class matching any single letter or digit; + requires one or more, so an empty string never validates. fullmatch() is stricter than search() — it demands the whole string, start to finish, satisfy the pattern, making it the right choice for validation tasks where a partial match shouldn't count as valid.

## Exercise 2. Match Zero or More

**Concept:** the * quantifier

**Problem:** Match a string that's an 'a' followed by zero or more 'b's.

**Given:**
```
test_strings = ["a", "ab", "abb", "abbb", "b", "ba"]
```

**Expected Output:**
```
a, ab, abb, abbb -> Match; b, ba -> No match
```

**Hint:** ab* means the b is entirely optional and can repeat any number of times, including zero.

In [ ]:
import re

pattern = r"ab*"
test_strings = ["a", "ab", "abb", "abbb", "b", "ba"]
for s in test_strings:
    result = re.fullmatch(pattern, s)
    print(f"{s:<6} -> {'Match' if result else 'No match'}")

**Explanation:** ab* matches a literal 'a' followed by zero or more 'b' characters — the * makes the b entirely optional while still allowing unlimited repeats. fullmatch() is essential here: without it, re.search() would still match the 'a' inside 'ba', producing a false positive. 'b' fails since there's no leading 'a'; 'ba' fails since the letters are in the wrong order.

## Exercise 3. Match One or More

**Concept:** the + quantifier

**Problem:** Match a string that's an 'a' followed by one or more 'b's.

**Given:**
```
test_strings = ["a", "ab", "abb", "abbb", "b", "ba"]
```

**Expected Output:**
```
ab, abb, abbb -> Match; a, b, ba -> No match
```

**Hint:** + requires at least one b, which is the key difference from Exercise 2's *.

In [ ]:
import re

pattern = r"ab+"
test_strings = ["a", "ab", "abb", "abbb", "b", "ba"]
for s in test_strings:
    result = re.fullmatch(pattern, s)
    print(f"{s:<6} -> {'Match' if result else 'No match'}")

**Explanation:** ab+ requires at least one 'b' after the 'a', unlike * which allowed zero. This is why 'a' alone — which matched in Exercise 2 — fails here: with + there must be at least one b present. The single-character swap from * to + is the entire difference between 'optional repetition' and 'required repetition'.

## Exercise 4. Match Optional Characters

**Concept:** the ? quantifier

**Problem:** Match a string that's exactly 'a' or 'ab', nothing else.

**Given:**
```
test_strings = ["a", "ab", "abb", "abbb", "b", "ba"]
```

**Expected Output:**
```
a, ab -> Match; abb, abbb, b, ba -> No match
```

**Hint:** ? means zero or ONE occurrence — unlike * and +, it never allows repetition.

In [ ]:
import re

pattern = r"ab?"
test_strings = ["a", "ab", "abb", "abbb", "b", "ba"]
for s in test_strings:
    result = re.fullmatch(pattern, s)
    print(f"{s:<6} -> {'Match' if result else 'No match'}")

**Explanation:** ab? matches 'a' followed by at most one 'b' — the ? permits zero or one occurrence but never more. 'abb' fails because two b's exceed that limit. The three basic quantifiers form a spectrum: ? is 0-1, * is 0-to-infinity, + is 1-to-infinity. A very common real-world use of ? is 'https?', which matches both 'http' and 'https'.

## Exercise 5. Match Exact Occurrences

**Concept:** the {n} curly-brace quantifier

**Problem:** Match a string that's an 'a' followed by exactly three 'b's.

**Given:**
```
test_strings = ["a", "ab", "abb", "abbb", "abbbb", "b"]
```

**Expected Output:**
```
Only abbb -> Match; everything else -> No match
```

**Hint:** {3} is equivalent to writing 'bbb' explicitly, but more readable and adjustable.

In [ ]:
import re

pattern = r"ab{3}"
test_strings = ["a", "ab", "abb", "abbb", "abbbb", "b"]
for s in test_strings:
    result = re.fullmatch(pattern, s)
    print(f"{s:<6} -> {'Match' if result else 'No match'}")

**Explanation:** b{3} matches exactly three b's — {n} applies only to the single element immediately before it (b here), not the whole preceding pattern. 'abbbb' fails because fullmatch() demands the entire string match, and a fourth b leaves extra unmatched content. The range form {m,n} (covered in the next exercise) generalizes this to a bounded interval rather than one exact count.

## Exercise 6. Match Range of Occurrences

**Concept:** the {m,n} range quantifier

**Problem:** Match a string that's an 'a' followed by two to three 'b's.

**Given:**
```
test_strings = ["a", "ab", "abb", "abbb", "abbbb", "b"]
```

**Expected Output:**
```
abb, abbb -> Match; everything else -> No match
```

**Hint:** {m,n} is inclusive on both ends — no space allowed between the comma and the numbers.

In [ ]:
import re

pattern = r"ab{2,3}"
test_strings = ["a", "ab", "abb", "abbb", "abbbb", "b"]

for s in test_strings:
    result = re.fullmatch(pattern, s)
    print(f"{s:<6} -> {'Match' if result else 'No match'}")

**Explanation:** b{2,3} matches b repeated 2 or 3 times, inclusive at both bounds. 'ab' fails since a single b falls below the minimum of two; 'abbbb' fails since four b's exceed the maximum of three. {2,3} sits between an exact count ({3}) and an open-ended minimum ({2,}, meaning 'two or more') — choose based on how tightly the input needs to be constrained.

## Exercise 7. Find Underscore Joined Lowercase

**Concept:** combining character classes with a literal separator

**Problem:** Match snake_case-style tokens: lowercase letters, underscore, lowercase letters.

**Given:**
```
test_strings = ["hello_world", "foo_bar", "hello", "hello_", "_world", "Hello_world", "hello_World"]
```

**Expected Output:**
```
hello_world, foo_bar -> Match; everything else -> No match
```

**Hint:** [a-z]+ requires at least one lowercase letter on EACH side of the underscore.

In [ ]:
import re

pattern = r"[a-z]+_[a-z]+"
test_strings = ["hello_world", "foo_bar", "hello", "hello_", "_world", "Hello_world", "hello_World"]

for s in test_strings:
    result = re.fullmatch(pattern, s)
    print(f"{s:<12} -> {'Match' if result else 'No match'}")

**Explanation:** [a-z]+_[a-z]+ requires one or more lowercase letters, a literal underscore, then one or more lowercase letters. 'hello_' and '_world' fail because one side of the underscore has nothing to satisfy its + requirement. 'Hello_world' and 'hello_World' fail because [a-z] only matches lowercase — any uppercase letter anywhere breaks the fullmatch().

## Exercise 8. PascalCase Match

**Concept:** positional character-class rules

**Problem:** Match a single uppercase letter followed by one or more lowercase letters.

**Given:**
```
test_strings = ["Hello", "World", "python", "HELLO", "Hello123", "H", "Ha"]
```

**Expected Output:**
```
Hello, World, Ha -> Match; python, HELLO, Hello123, H -> No match
```

**Hint:** [A-Z] with no quantifier means exactly one uppercase letter — not zero, not two.

In [ ]:
import re

pattern = r"[A-Z][a-z]+"
test_strings = ["Hello", "World", "python", "HELLO", "Hello123", "H", "Ha"]
for s in test_strings:
    result = re.fullmatch(pattern, s)
    print(f"{s:<8} -> {'Match' if result else 'No match'}")

**Explanation:** [A-Z] (no quantifier) matches exactly one uppercase letter, and [a-z]+ then requires at least one lowercase letter after it. 'H' fails because the + demands at least one lowercase letter to follow the capital, and there isn't one. 'HELLO' fails because after matching the leading 'H', the remaining 'ELLO' is all uppercase — [a-z]+ finds nothing to match. 'Hello123' fails because fullmatch() requires the trailing digits to be consumed too, and nothing in the pattern accounts for them.

## Exercise 9. Match Start and End

**Concept:** the dot wildcard combined with .* for 'anything in between'

**Problem:** Match a string that starts with 'a', ends with 'b', with any characters in between.

**Given:**
```
test_strings = ["a123b", "axyzb", "ab", "a b", "ab ", "b123a", "a123"]
```

**Expected Output:**
```
a123b, axyzb, ab, a b -> Match; ab (trailing space), b123a, a123 -> No match
```

**Hint:** fullmatch() with a.*b naturally anchors to the whole string, so explicit ^ and $ aren't needed.

In [ ]:
import re

pattern = r"a.*b"
test_strings = ["a123b", "axyzb", "ab", "a b", "ab ", "b123a", "a123"]

for s in test_strings:
    result = re.fullmatch(pattern, s)
    print(f"{s:<6} -> {'Match' if result else 'No match'}")

**Explanation:** The dot . matches any single character except a newline (not a literal period — that would need \.). .* combines it with zero-or-more repetition, allowing the middle section to be empty, one character, or arbitrarily long — which is why plain 'ab' matches (the middle matches zero characters). The trailing-space version of 'ab ' fails because fullmatch() demands every character be consumed, and that trailing space isn't accounted for by the pattern.

## Exercise 10. Match Word at Start

**Concept:** the ^ anchor + \b word boundary

**Problem:** Match a specific word only when it appears at the very start of a string.

**Given:**
```
test_strings = ["Hello world", "Hello", "Say Hello", "hello world", "HelloWorld"]
```

**Expected Output:**
```
Hello world, Hello -> Match; Say Hello, hello world, HelloWorld -> No match
```

**Hint:** \b after the word prevents 'HelloWorld' from matching, since there's no word boundary between 'Hello' and 'World'.

In [ ]:
import re

pattern = r"^Hello\b"
test_strings = ["Hello world", "Hello", "Say Hello", "hello world", "HelloWorld"]
for s in test_strings:
    result = re.search(pattern, s)
    print(f"{s:<12} -> {'Match' if result else 'No match'}")

**Explanation:** ^ asserts (without consuming any characters) that what follows must begin at position 0 of the string. \b is a word-boundary assertion — the position between a word character and a non-word character — which is what correctly rejects 'HelloWorld', since there's no such boundary between the 'o' and the 'W'. re.search() (rather than fullmatch()) is used because the string may legitimately have more content after the matched word, as in 'Hello world'.

## Exercise 11. Match Word at End

**Concept:** the $ anchor + an optional trailing-punctuation class

**Problem:** Match a word at the end of a string, allowing one optional trailing punctuation mark.

**Given:**
```
test_strings = ["I love Python", "Python is great", "I love Python!", "python", "I love Python."]
```

**Expected Output:**
```
I love Python, I love Python!, I love Python. -> Match; Python is great, python -> No match
```

**Hint:** [.,!?]? makes the trailing punctuation optional — present or absent, both are accepted.

In [ ]:
import re

pattern = r"\bPython[.,!?]?$"
test_strings = ["I love Python", "Python is great", "I love Python!", "python", "I love Python."]

for s in test_strings:
    result = re.search(pattern, s)
    print(f"{s:<16} -> {'Match' if result else 'No match'}")

**Explanation:** \b before 'Python' ensures the match starts at a genuine word edge, not partway through a longer word like 'CPython'. [.,!?]? allows at most one trailing punctuation character before the required $ end-of-string anchor. 'Python is great' fails because 'Python' isn't at the end; 'python' (lowercase) fails because regex matching is case-sensitive by default.

## Exercise 12. Find a Specific Letter

**Concept:** \w* on both sides of a target character, with \b boundaries

**Problem:** Find all words in a sentence that contain the letter 'z' anywhere.

**Given:**
```
text = "The pizza was amazing but the fizz and buzz were too loud"
```

**Expected Output:**
```
['pizza', 'amazing', 'fizz', 'buzz']
```

**Hint:** \w* (zero or more) on both sides lets z be at the very start or end of the word too, as in 'fizz'/'buzz'.

In [ ]:
import re

text = "The pizza was amazing but the fizz and buzz were too loud"
pattern = r"\b\w*z\w*\b"

matches = re.findall(pattern, text)
print(matches)

**Explanation:** \w matches any word character (letters, digits, underscore); \w* on each side of the literal z allows zero or more of them, which is why words where z sits right at the start or end (like 'fizz', 'buzz') are still captured. The \b boundaries on both ends ensure only complete words are returned, not arbitrary substrings. findall() scans left to right and returns every non-overlapping match as a list.

## Exercise 13. Find Letter in Middle

**Concept:** switching \w* to \w+ to require characters on both sides

**Problem:** Find words containing 'z', but only where z is strictly interior (not first or last letter).

**Given:**
```
text = "The pizza was amazing but the fizz and buzz were too loud"
```

**Expected Output:**
```
['pizza', 'amazing']
```

**Hint:** Swapping * for + on both sides is the ONLY change from Exercise 12 — and it changes the meaning entirely.

In [ ]:
import re

text = "The pizza was amazing but the fizz and buzz were too loud"
pattern = r"\b\w+z\w+\b"

matches = re.findall(pattern, text)
print(matches)

**Explanation:** Requiring \w+ (one or more, not zero or more) on both sides of the z forces at least one character to exist both before AND after it — pushing z strictly into the interior of the word. 'fizz' and 'buzz' are now excluded because their z sits at the very end, leaving nothing left over to satisfy the trailing \w+. This single quantifier swap is the entire difference in behavior from the previous exercise.

## Exercise 14. Match Adjacent Words

**Concept:** chaining two word-patterns with \s+ between them

**Problem:** Match two consecutive words that both start with capital P.

**Given:**
```
test_strings = ["Peter Parker is here", "Paul and Peter met", "Pretty Please", "Python Programming is fun", "No match here"]
```

**Expected Output:**
```
Peter Parker, Pretty Please, Python Programming -> Match; others -> No match
```

**Hint:** \s+ (not a literal space) between the two P-words handles multiple spaces or tabs robustly.

In [ ]:
import re

pattern = r"P\w*\s+P\w*"
test_strings = [
    "Peter Parker is here",
    "Paul and Peter met",
    "Pretty Please",
    "Python Programming is fun",
    "No match here"
]

for s in test_strings:
    result = re.search(pattern, s)
    if result:
        print(f"{s:<26} -> Match: {result.group()}")
    else:
        print(f"{s:<26} -> No match")

**Explanation:** P\w* matches any word starting with uppercase P, and appears twice, separated by \s+ for one-or-more whitespace characters (robust against multiple spaces or a tab). 'Paul and Peter met' fails despite both words starting with P, because the word 'and' sits between them, breaking the required adjacency. result.group() returns the exact substring matched, useful for confirming which specific pair was found.

## Exercise 15. Filter by Starting Letter

**Concept:** a character class as a multi-option first character

**Problem:** Find all words in a sentence that start with 'a' or 'e'.

**Given:**
```
text = "an eagle soared above the endless empty arena every afternoon"
```

**Expected Output:**
```
['an', 'eagle', 'above', 'endless', 'empty', 'arena', 'every', 'afternoon']
```

**Hint:** [ae] is more concise than alternation (a|e) for single-character options.

In [ ]:
import re

text = "an eagle soared above the endless empty arena every afternoon"
pattern = r"\b[ae]\w*"

matches = re.findall(pattern, text)
print(matches)

**Explanation:** \b anchors the match to the START of a word specifically, so an 'a' or 'e' appearing in the MIDDLE of a longer word wouldn't trigger a match. [ae] matches either letter as the required first character, and \w* (zero or more) captures the rest of the word, including the case where the word is just a single letter like 'an' minus its n — though here 'an' matches in full since \w* greedily consumes as much as it can.

## Exercise 16. Validate Alphanumeric ID

**Concept:** \w+ as a strict allowlist pattern

**Problem:** Validate that a string contains only letters, digits, and underscores — nothing else.

**Given:**
```
test_strings = ["user_123", "User_Name", "invalid id", "bad-char!", "_leadingUnderscore", "ALL_CAPS_99"]
```

**Expected Output:**
```
user_123, User_Name, _leadingUnderscore, ALL_CAPS_99 -> Valid; others -> Invalid
```

**Hint:** \w is shorthand for [a-zA-Z0-9_] — it already excludes spaces, hyphens, and punctuation with no extra work.

In [ ]:
import re

pattern = r"\w+"
test_strings = ["user_123", "User_Name", "invalid id", "bad-char!", "_leadingUnderscore", "ALL_CAPS_99"]

for s in test_strings:
    result = re.fullmatch(pattern, s)
    print(f"{s:<20} -> {'Valid' if result else 'Invalid'}")

**Explanation:** \w is exactly equivalent to the character class [a-zA-Z0-9_] — any letter, digit, or underscore — and matches nothing else, so spaces, hyphens, and punctuation are automatically excluded with no extra pattern needed. fullmatch() then enforces that every single character in the string belongs to that set; one disallowed character anywhere fails the whole check. A leading underscore is valid since _ is part of \w, mirroring Python's own identifier naming rules.

## Exercise 17. Check Starting Number

**Concept:** ^ anchor combined with an f-string-built pattern and \b

**Problem:** Verify a string starts with a specific target number.

**Given:**
```
test_strings = ["42 is the answer", "42", "The answer is 42", "420 wide", "142 steps"], target = "42"
```

**Expected Output:**
```
42 is the answer, 42 -> Match; others -> No match
```

**Hint:** \b after the target number prevents '42' from matching inside a longer number like '420'.

In [ ]:
import re

target = "42"
pattern = rf"^{target}\b"
test_strings = ["42 is the answer", "42", "The answer is 42", "420 wide", "142 steps"]

for s in test_strings:
    result = re.search(pattern, s)
    print(f"{s:<17} -> {'Match' if result else 'No match'}")

**Explanation:** The rf-prefixed string combines raw-string behavior (needed for \b) with f-string interpolation (needed to insert the target variable), making the pattern reusable for any target number without hand-editing the regex. ^ anchors to the very start of the string, and \b afterward prevents '42' from matching just the first two digits of '420' — without it, '420 wide' would be wrongly reported as a match.

## Exercise 18. Number at End

**Concept:** \d+ combined with the $ anchor

**Problem:** Check whether a string ends with one or more digits.

**Given:**
```
test_strings = ["version 2", "file_backup_3", "hello", "order 99b", "track5", "2024"]
```

**Expected Output:**
```
version 2, file_backup_3, track5, 2024 -> Ends with a number; hello, order 99b -> Does not
```

**Hint:** \d+ (not just \d) captures the full trailing digit run, useful if you later want to extract the value.

In [ ]:
import re

pattern = r"\d+$"
test_strings = ["version 2", "file_backup_3", "hello", "order 99b", "track5", "2024"]

for s in test_strings:
    result = re.search(pattern, s)
    if result:
        print(f"{s:<14} -> Ends with a number")
    else:
        print(f"{s:<14} -> Does not end with a number")

**Explanation:** \d is shorthand for [0-9]; \d+ matches one or more consecutive digits, capturing the full trailing numeric run rather than just the final digit (useful if you later call result.group() to extract the value). $ anchors the digit run to the very last position of the string. 'order 99b' fails because the string ends with the letter 'b', not a digit — even though digits appear earlier, $ specifically requires the FINAL character to satisfy \d.

## Exercise 19. Clean IP Addresses

**Concept:** re.sub() with a callable replacement function

**Problem:** Strip leading zeros from each numeric segment of an IP address.

**Given:**
```
ip_addresses = ["192.168.001.001", "010.000.000.001", "255.255.255.000", "192.168.1.1"]
```

**Expected Output:**
```
192.168.001.001 -> 192.168.1.1
010.000.000.001 -> 10.0.0.1
```

**Hint:** Passing a lambda as re.sub()'s replacement receives the match object and lets you compute the replacement.

In [ ]:
import re

ip_addresses = ["192.168.001.001", "010.000.000.001", "255.255.255.000", "192.168.1.1"]

def remove_leading_zeros(ip):
    return re.sub(r"\d+", lambda m: str(int(m.group())), ip)

for ip in ip_addresses:
    cleaned = remove_leading_zeros(ip)
    print(f"{ip:<16} -> {cleaned}")

**Explanation:** re.sub() finds every match of \d+ (each numeric segment, since the dots aren't digits) and, because the replacement argument is callable rather than a plain string, calls it once per match with the match object. m.group() retrieves the matched text ('001'), int() converts it (dropping leading zeros), and str() converts it back for substitution. Segments that already have no leading zeros round-trip unchanged through int()/str(), so already-clean input like '192.168.1.1' passes through safely.

## Exercise 20. Convert Date Format

**Concept:** capturing groups + backreferences in re.sub()

**Problem:** Convert a date from yyyy-mm-dd to dd-mm-yyyy format.

**Given:**
```
dates = ["2024-01-15", "1999-12-31", "2000-07-04", "2024-11-05"]
```

**Expected Output:**
```
2024-01-15 -> 15-01-2024
```

**Hint:** \1, \2, \3 in the replacement string refer back to the three parenthesized capturing groups in order.

In [ ]:
import re

dates = ["2024-01-15", "1999-12-31", "2000-07-04", "2024-11-05"]
pattern = r"(\d{4})-(\d{2})-(\d{2})"
replacement = r"\3-\2-\1"

for date in dates:
    converted = re.sub(pattern, replacement, date)
    print(f"{date} -> {converted}")

**Explanation:** Each (\d{...}) parenthesized group both matches its part of the date AND captures the matched text for later reference: group 1 is the year, group 2 the month, group 3 the day. The backreferences \1, \2, \3 in the replacement string let you rearrange that captured text into a new order (\3-\2-\1 = day-month-year) with zero manual string slicing. Because the groups capture raw digit text rather than converting to integers, zero-padding like '01' or '07' is preserved exactly through the substitution.

## Exercise 21. Extract 1-3 Digit Numbers

**Concept:** \d{1,3} bounded by \b to exclude longer numbers entirely

**Problem:** Extract only the numbers in a text that are 1 to 3 digits long, excluding longer ones.

**Given:**
```
text = "There are 3 cats, 12 dogs, 500 fish, 1000 birds, and 42 turtles in the sanctuary"
```

**Expected Output:**
```
['3', '12', '500', '42']
```

**Hint:** Without \b boundaries, \d{1,3} would grab just the first 3 digits of '1000', producing a wrong partial match.

In [ ]:
import re

text = "There are 3 cats, 12 dogs, 500 fish, 1000 birds, and 42 turtles in the sanctuary"
pattern = r"\b\d{1,3}\b"

matches = re.findall(pattern, text)
print(matches)

**Explanation:** \d{1,3} alone would greedily match up to 3 digits anywhere, including within a longer digit run. The \b boundaries on both sides are what make this exercise work correctly: they require the digit run to be bordered by non-digit characters on each side, so '1000' is excluded ENTIRELY rather than being incorrectly truncated to '100' — the closing \b can't find a valid boundary between two digit characters, so the whole match attempt fails for that token.

## Exercise 22. Search Literal Strings

**Concept:** the alternation operator | combined with re.finditer()

**Problem:** Search for either of two specific words and report where each is found.

**Given:**
```
text = "The quick brown fox jumps over the lazy dog", targets = fox, dog
```

**Expected Output:**
```
Found "fox" at index 16-19
Found "dog" at index 40-43
```

**Hint:** Parentheses around fox|dog matter — without them, boundary markers outside would bind incorrectly.

In [ ]:
import re

text = "The quick brown fox jumps over the lazy dog"
pattern = r"\b(fox|dog)\b"

for match in re.finditer(pattern, text):
    print(f'Found "{match.group()}" at index {match.start()}-{match.end()}')

**Explanation:** fox|dog tries 'fox' first at each position, then 'dog' if that fails — alternatives are evaluated left to right, and any number of them can be chained with more |. Wrapping them in (fox|dog) matters: without parentheses, \bfox|dog\b would actually parse as (\bfox) OR (dog\b), breaking the intended boundary behavior for both words. finditer() returns full match objects (not just strings), giving access to .start() and .end() for exact positional info.

## Exercise 23. Find Pattern Location

**Concept:** re.search() + .start()/.end()/.span() for positional info

**Problem:** Find a literal phrase and report its exact start and end index.

**Given:**
```
text = "The quick brown fox jumps over the lazy dog", target = "brown fox"
```

**Expected Output:**
```
Found "brown fox" at start=10, end=19
```

**Hint:** re.escape() neutralizes any regex-special characters in the target — a good habit for any non-hardcoded search term.

In [ ]:
import re

text = "The quick brown fox jumps over the lazy dog"
target = "brown fox"
pattern = re.escape(target)

match = re.search(pattern, text)
if match:
    print(f'Found "{match.group()}" at start={match.start()}, end={match.end()}')
else:
    print(f'"{target}" not found in the text')

**Explanation:** re.escape(target) escapes any characters with special regex meaning (., *, +, (, etc.) in the target string — invisible here since 'brown fox' has none, but essential whenever a search term comes from user input where such characters can't be ruled out. match.start() and match.end() give the zero-based start (inclusive) and end (exclusive) positions, following Python's normal slicing convention — match.span() returns both as a single (start, end) tuple if that's more convenient.

## Exercise 24. Find All Substrings

**Concept:** re.findall() for raw substring counting, deliberately without \b

**Problem:** Find every occurrence of a substring, including ones embedded inside longer words.

**Given:**
```
text = "cat and cattle and catfish and catch and tomcat", target = "cat"
```

**Expected Output:**
```
['cat', 'cat', 'cat', 'cat', 'cat'] — Total count: 5
```

**Hint:** This is intentionally a raw substring search — \b word boundaries are deliberately omitted here.

In [ ]:
import re

text = "cat and cattle and catfish and catch and tomcat"
target = "cat"
pattern = re.escape(target)

matches = re.findall(pattern, text)
print(f'Occurrences of "{target}": {matches}')
print(f"Total count: {len(matches)}")

**Explanation:** Unlike Exercise 12, this pattern deliberately has no \b boundaries, so it captures 'cat' wherever it appears — as a standalone word, as a prefix ('cattle', 'catfish', 'catch'), or as a suffix ('tomcat'). When the pattern has no capturing groups, findall() returns the matched substrings themselves as a plain list; len() of that list directly gives the occurrence count, equivalent to text.count(target) here but generalizing to more complex patterns where str.count() can't help.

## Exercise 25. Iterate Matches

**Concept:** re.finditer() for positions alongside matched text

**Problem:** Find every occurrence of a substring along with its exact position.

**Given:**
```
text = "cat and cattle and catfish and catch and tomcat", target = "cat"
```

**Expected Output:**
```
Match 1: "cat" found at position 0-3
Match 2: "cat" found at position 8-11 ...
```

**Hint:** finditer() is a lazy iterator (memory-efficient for large text), unlike findall() which builds the full list upfront.

In [ ]:
import re

text = "cat and cattle and catfish and catch and tomcat"
target = "cat"
pattern = re.escape(target)

for i, match in enumerate(re.finditer(pattern, text), start=1):
    print(f'Match {i}: "{match.group()}" found at position {match.start()}-{match.end()}')

**Explanation:** finditer() returns a lazy iterator of match objects rather than materializing every result into a list upfront, which matters for memory usage on large texts. enumerate(..., start=1) numbers the matches starting from 1 for natural-reading output. The two functions are complementary: use findall() when you only need the matched values, use finditer() when you also need positional information or want to process matches one at a time.

## Exercise 26. Extract Date from URL

**Concept:** multiple capturing groups + match.groups()

**Problem:** Extract year, month, and day from a URL formatted as /yyyy/mm/dd/.

**Given:**
```
urls containing paths like /2026/05/22/my-article
```

**Expected Output:**
```
Year: 2026 | Month: 05 | Day: 22
```

**Hint:** Leading/trailing slashes in the pattern anchor the date segment specifically within the URL path structure.

In [ ]:
import re

urls = [
    "https://example.com/2026/05/22/my-article",
    "https://news.site.org/2019/11/03/breaking-story",
    "https://blog.example.com/2023/07/30/summer-update"
]

pattern = r"/(\d{4})/(\d{2})/(\d{2})/"

for url in urls:
    match = re.search(pattern, url)
    if match:
        year, month, day = match.groups()
        print(f"URL: {url}")
        print(f"  Year: {year} | Month: {month} | Day: {day}\n")

**Explanation:** The forward slashes surrounding each capturing group anchor the date segments specifically within the URL's path structure, preventing an accidental match against, say, a numeric slug elsewhere. Three groups — (\d{4}), (\d{2}), (\d{2}) — isolate year, month, and day respectively, with the fixed-width quantifiers matching the exact expected format. match.groups() returns all captured groups as one tuple in pattern order, which unpacks cleanly into year, month, day = match.groups().

## Exercise 27. Extract All Numbers

**Concept:** \d+\.?\d* to capture both integers and decimals

**Problem:** Extract every numeric value — both whole numbers and decimals — from mixed text.

**Given:**
```
text = "In 2024 there were 1200 participants across 3 events, with scores of 98.5, 76, and 100"
```

**Expected Output:**
```
['2024', '1200', '3', '98.5', '76', '100']
```

**Hint:** \.? makes the decimal point optional, so pure integers and decimal numbers are both matched by one pattern.

In [ ]:
import re

text = "In 2024 there were 1200 participants across 3 events, with scores of 98.5, 76, and 100"
pattern = r"\d+\.?\d*"

matches = re.findall(pattern, text)
print(matches)

**Explanation:** \d+ matches the required leading digit run (handling plain integers like '2024' and also anchoring the integer part of a decimal like '98.5'). \.? makes the decimal point itself optional — needed since escaping with backslash is required, as an unescaped dot means 'any character'. \d* (zero or more) allows for digits after the optional dot, or none at all. Results from findall() always come back as strings; converting them to numbers requires an explicit [float(n) for n in matches].

## Exercise 28. Extract Email Addresses

**Concept:** a multi-part pattern mirroring email's real structural rules

**Problem:** Extract every valid-looking email address from a block of text, rejecting malformed ones.

**Given:**
```
text containing valid emails plus invalid fragments like @nodomain and user@
```

**Expected Output:**
```
['support@example.com', 'admin.team@company.org', 'sales@shop.co.uk', 'billing_dept+invoices@finance.example.net']
```

**Hint:** A non-capturing group (?:...) groups an alternative without affecting what findall() returns.

In [ ]:
import re

text = """Please reach out to support@example.com for help.
You can also contact the team at admin.team@company.org or sales@shop.co.uk.
Invalid addresses like @nodomain and user@ should be ignored.
For billing queries write to billing_dept+invoices@finance.example.net."""

pattern = r"[\w.+\-]+@[\w\-]+(?:\.[\w\-]+)*\.[a-zA-Z]{2,}"

matches = re.findall(pattern, text)
print(matches)

**Explanation:** [\w.+\-]+ matches the local part (before @) — word characters plus dots, plus signs, and hyphens, requiring at least one character, which is why '@nodomain' (nothing before @) is correctly rejected. @ is a mandatory literal separator. [\w\-]+ matches the first domain label, rejecting 'user@' since nothing follows. (?:\.[\w\-]+)* is a non-capturing group (the ?: prefix means it groups without capturing, so findall() still returns the full email rather than a group-only tuple) allowing zero or more additional dot-separated subdomain labels. Finally \.[a-zA-Z]{2,} demands a proper top-level domain of at least 2 letters.

## Exercise 29. Swap Characters

**Concept:** a single-pass conditional substitution to avoid the classic swap bug

**Problem:** Replace every space with an underscore AND every underscore with a space, simultaneously.

**Given:**
```
test_strings = ["hello world", "hello_world", "the quick_brown fox_jumps", "no_change"]
```

**Expected Output:**
```
hello world -> hello_world; hello_world -> hello world
```

**Hint:** Two SEQUENTIAL re.sub() calls would fail — the second call would undo part of the first.

In [ ]:
import re

test_strings = [
    "hello world",
    "hello_world",
    "the quick_brown fox_jumps",
    "no_change"
]

def swap(s):
    return re.sub(r"[ _]", lambda m: "_" if m.group() == " " else " ", s)

for s in test_strings:
    print(f"{s:<26} -> {swap(s)}")

**Explanation:** Running two sequential re.sub() calls (spaces->underscores, then underscores->spaces) would fail: the second call would also convert the underscores the first call just inserted, undoing the swap entirely. The single-pass fix uses a character class [ _] to catch either character, then a lambda replacement that decides — per individual match, before any text is rewritten — which character to substitute in. Because each decision is made independently at scan time, there's no risk of a freshly-substituted character being caught and flipped again.

## Exercise 30. Replace Multiple Delimiters

**Concept:** a character class replacing several different delimiters with one call

**Problem:** Replace spaces, commas, and dots all with a single colon.

**Given:**
```
test_strings = ["one two three", "one,two,three", "one.two.three", "one, two. three"]
```

**Expected Output:**
```
one two three -> one:two:three; one, two. three -> one::two::three
```

**Hint:** Inside a character class, a dot loses its 'any character' meaning and becomes literal — no escaping needed.

In [ ]:
import re

test_strings = [
    "one two three",
    "one,two,three",
    "one.two.three",
    "one, two. three",
    "no.delimiters,here today"
]

pattern = r"[ ,.]"

for s in test_strings:
    result = re.sub(pattern, ":", s)
    print(f"{s:<26} -> {result}")

**Explanation:** [ ,.] is a character class matching any ONE of a space, comma, or dot — and inside [...], the dot is just a literal character, not a wildcard, so no backslash escaping is needed. Since the replacement is a plain string ":", every matched delimiter becomes a colon with no lambda required. 'one, two. three' produces a double colon '::' because the comma and the following space are each independently matched and replaced — this is expected behavior; changing the pattern to [ ,.]+ would instead collapse consecutive delimiters into a single colon.